# Chapter 5 — Key Learnings

This notebook contains my main takeaways, definitions, and conceptual notes
from **Chapter 4 - Pretraining on unlabeled daat** of *'Build a Large Language Model (From Scratch)'* book by Sebastian Raschka.

### 0. Chapter Objective

Pretrain the GPT model using next-token prediction, evaluate its loss, generate text with controlled randomness, and save/reload learned weights.

### 1. Adding our repo root 'build-llm-from-scratch-pytorch' to sys.path

In [1]:
from pathlib import Path
import sys

# Current folder:
# repository_root/chapter_04/exercises
# i.e. 
# import os  
# print(os.getcwd()) # prints: c:\Users\delmi\Documents\LEARNING\Manning_Learning\repos\build-llm-from-scratch-pytorch\chapter_04\exercises 
# Note: Python searches for chapter_03 (and all other needed imports) inside that folder and in the other locations listed in sys.path. but sys.path currently does not have the repo root
# the snippet below adds the repo root to sys.path

repo_root = Path.cwd().parents[1]

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repository root:", repo_root)
sys.path

Repository root: c:\Users\delmi\Documents\LEARNING\Manning_Learning\repos\build-llm-from-scratch-pytorch


['c:\\Users\\delmi\\Documents\\LEARNING\\Manning_Learning\\repos\\build-llm-from-scratch-pytorch',
 'C:\\Users\\delmi\\anaconda3\\python312.zip',
 'C:\\Users\\delmi\\anaconda3\\DLLs',
 'C:\\Users\\delmi\\anaconda3\\Lib',
 'C:\\Users\\delmi\\anaconda3',
 'c:\\Users\\delmi\\venvs\\llmbookvenv',
 '',
 'c:\\Users\\delmi\\venvs\\llmbookvenv\\Lib\\site-packages']

### 2. Probabilistic Next-Token Sampling

After the model produces logits:

```text
logits
→ optional Top-K filtering
→ temperature scaling
→ softmax
→ probability distribution
→ sample next token
```


**Binomial and Multinomial distributions and how they relate to top-k and probability sampling in LLMs**
**Binomial distribution:** Models the number of successes in a fixed number of independent trials, where each trial has two possible outcomes.

- **Input:** Number of trials $n$, success probability $p$
- **Output:** Probability of getting $k$ successes, where $k = 0, \dots, n$

```math
P(X=k)=\binom{n}{k}p^k(1-p)^{n-k}
```

**Multinomial distribution:** Generalizes the binomial distribution to more than two possible outcomes per trial.

- **Input:** Number of trials $n$, category probabilities $(p_1, \dots, p_K)$
- **Output:** Probability of getting counts $(k_1, \dots, k_K)$, where $\sum_{i=1}^{K} k_i = n$

```math
P(X_1=k_1,\dots,X_K=k_K)
=
\frac{n!}{k_1!\cdots k_K!}
\prod_{i=1}^{K} p_i^{k_i}
```


**Crisp intuition:**

> **Binomial:** “How many successes out of $n$?”  
> **Multinomial:** “How many times did each of $K$ categories occur out of $n$?”

**Relation to top-k sampling in LLMs:** Top-k sampling first keeps only the $k$ highest-probability next tokens and renormalizes their probabilities so they sum to 1. The model then *samples one token from this probability distribution*.

> **Top-k decides which tokens are allowed; multinomial sampling decides which allowed token is actually chosen, according to their probabilities.**


Summary

- **Binomial:** two possible outcomes.
- **Multinomial:** multiple possible outcomes.
- An LLM chooses among thousands of vocabulary tokens, so next-token sampling uses a multinomial/categorical distribution.

`torch.multinomial(probs, 1)` samples one token according to its probability.

 Temperature

`logits / temperature`

- Lower temperature → sharper distribution → more deterministic.
- Higher temperature → flatter distribution → more diverse.

### Top-K

Keep only the `K` highest-scoring tokens and set all other logits to `-inf` before softmax.

**Together:** Top-K restricts *which tokens may be sampled*; temperature changes *how probability is distributed among them*.

> **Terminology note:** A single next-token draw is most precisely a **categorical** sample; PyTorch exposes this through `torch.multinomial`.


### 3. calc_loss_batch() and calc_loss_loader() functions

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device): # calculates the loss for a single batch
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0,1), target_batch.flatten() #first argument is the predicted labels, second argument is the actual labels
    )
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None): # num_batches can be set as a smaller number if we want to speedup evaluation during training
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches == None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
    return loss / num_batches

**Purpose:** Computes next-token cross-entropy loss for one batch.

```text
logits:  [batch, tokens, vocab_size]
         ↓ flatten
         [batch × tokens, vocab_size]

targets: [batch, tokens]
         ↓ flatten
         [batch × tokens]
```


This shape transformation is one of the things worth remembering.


### 4. train_model_simple() and evaluate_model() functions 

In [3]:
import torch
from chapter_05.scripts.generation import generate_and_print_sample

def train_model_simple(model, train_loader, val_loader,
                       optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(
                input_batch, target_batch, model, device
            )
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, "
                      f"Val loss {val_loss:.3f}"
                )

        generate_and_print_sample(
            model, tokenizer, device, start_context
        )
    return train_losses, val_losses, track_tokens_seen

def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(
            train_loader, model, device, num_batches=eval_iter
        )
        val_loss = calc_loss_loader(
            val_loader, model, device, num_batches=eval_iter
        )
    model.train()
    return train_loss, val_loss

Training Loop Mental Model

```text
batch
→ forward pass
→ cross-entropy loss
→ zero gradients
→ backward()
→ optimizer.step()
→ periodically evaluate train/validation loss
→ periodically generate a sample
```

### 5. generate() function

In [4]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens): # determines the number of iterations
        idx_cond = idx[:, -context_size:] # idx conditioned by the context
        with torch.no_grad():
            logits = model(idx_cond) # shape: (b, seq_len, vocab_size)
        logits = logits[:, -1, :] # we only care about the last token

        if top_k is not None: # i.e. top-k sampling
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(
                condition= logits < min_val,
                input= torch.tensor(float('-inf')).to(logits.device), # i.e. masking with -inf the non-topk values so that later the softmax only considers the top k values
                other= logits
            )

        if temperature > 0.0:
            logits = logits/temperature
            probs = torch.softmax(logits, dim=-1) # re-normalizing the probabilities so that they sum up to 1
            idx_next = torch.multinomial(probs, num_samples=1) # i.e. multinomial sampling - model samples one token from this probability distribution
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True) # i.e using greedy decoding sampling

        if idx_next == eos_id:
            break

        idx = torch.cat((idx, idx_next), dim=1)
    return idx    

### 6. Saving and Loading Model and Optimizer states - Checkpointing

```python
# SAVING Model only
torch.save(model.state_dict(), "../data/model.pth")

# LOADING Model only
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(torch.load("../data/model.pth", map_location=device))
model.eval() # Disables the dropout layers

# SAVING Model and Optimizer states
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict()
},
"../data/model_and_optimizer.pth")

# LOADING Model and Optimizer state
checkpoint = torch.load("../data/model_and_optimizer.pth", map_location=device)
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

model.train()
```

NOTE:  
- Save only the model state when the goal is inference.
- Save both model and optimizer states when training may resume.
- The optimizer state matters because AdamW maintains additional historical state used for subsequent updates.

### 7. Loading Pretrained GPT-2 Weights (from OpenAI)

Instead of performing expensive large-scale pretraining ourselves:

1. Download OpenAI's pretrained GPT-2 weights.
2. Configure `GPTModel` to match the corresponding GPT-2 architecture.
3. Map the downloaded weights into our PyTorch model.
4. Use the resulting pretrained model for generation or later fine-tuning.

Important: the architecture must match the pretrained weights, including
embedding size, number of layers, attention heads, context length, and QKV bias.

The downloaded weights replace the randomly initialized parameters of our
`GPTModel`.

### 8. Chapter 5 Flow

```text
Tokenized training data
        ↓
GPTModel
        ↓
Vocabulary logits
        ↓
Cross-entropy loss
        ↓
Backpropagation + AdamW
        ↓
Pretrained model
        ↓
Save checkpoint
        ↓
Generation
   ├── greedy
   └── temperature + Top-K sampling
```

### 9. Key definitions

- **Cross-entropy loss:** Measures how well the model's predicted next-token probability distribution matches the correct next token.

- **Greedy decoding:** Selects the highest-scoring token at every generation step.

- **Temperature:** Controls the sharpness of the next-token probability distribution during generation.

- **Top-K sampling:** Restricts sampling to the `K` highest-scoring candidate tokens.

- **Multinomial sampling:** Randomly selects a token according to the probability distribution over candidate tokens.
- **Checkpoint:** A saved model state, optionally including optimizer state, used for inference or resuming training.

### 10. Q/As
- **Q: What is the purpose of `calc_loss_batch()`?**  
  It computes the next-token cross-entropy loss for a single batch by comparing the model’s vocabulary logits with the target token IDs.

- **Q: What is the purpose of `calc_loss_loader()`?**  
  It evaluates the model over multiple batches and returns the average loss across those batches.

- **Q: What is the main purpose of `train_model_simple()`?**  
  It performs the LLM training loop: forward pass, loss calculation, backpropagation, optimizer update, periodic evaluation, and sample generation.

- **Q: What is the difference between greedy decoding and probabilistic sampling?**  
  Greedy decoding always selects the highest-scoring token, while probabilistic sampling draws a token according to the model’s probability distribution.

- **Q: What does temperature control during text generation?**  
  Temperature controls the sharpness of the next-token probability distribution: lower values make generation more deterministic, while higher values increase diversity.

- **Q: What is the purpose of Top-K sampling?**  
  Top-K sampling restricts generation to the `K` highest-scoring candidate tokens before sampling.

- **Q: Why save both the model state and optimizer state?**  
  The model state preserves learned weights, while the optimizer state preserves training statistics so training can resume properly.

- **Q: Why load pretrained GPT-2 weights instead of always pretraining from scratch?**  
  Large-scale pretraining is computationally expensive, so pretrained GPT-2 weights let us reuse a model that has already learned from a much larger corpus.